# 80 — Blind-A responder swap: Gemini-generated `predicted_response` -> submission zip

Takes the existing Blind-A prediction (whose `predicted_track_ids` came from the
validated retrieval+rerank pipeline, e.g. config 194 via colab/41) and regenerates
ONLY `predicted_response` with the Gemini API, then packages the CodaBench zip.
`predicted_track_ids` are left untouched -> nDCG/diversity axes unchanged; this only
moves the LLM axis (0.30 of the composite).

No GPU needed (no local responder, no retrieval). Prereqs: `GEMINI_API_KEY` in Colab
secrets, and an existing Blind-A `predicted_track_ids` prediction.json (from colab/41
or on Drive). top_n=1 by default (v5-kto's winning setting; not assumed optimal for
Gemini -- A/B 1 vs 3 on dev later via nb79's judge).

Rules note: external LLM APIs aren't banned, but final code must be uploaded
(due 2026-07-09) and the judge family is Gemini (self-preference risk). See
project_responder_topn_ab_status_2026_06_04 memory.


In [ ]:
# 1) Setup — clone branch + Gemini key + Drive + light deps (no GPU).
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'
from google.colab import userdata, drive
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')          # add this Colab secret first
os.environ.setdefault('GEMINI_RESPONDER_MODEL', 'gemini-2.5-flash-lite')   # lite = cheap+fast; gains come from structured_personality, not a heavier model (pro only gave +0.02)
drive.mount('/content/drive', force_remount=False)
BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!pip install -q -U google-generativeai 'datasets' 'pandas<3.0'
print('setup done | GEMINI key present:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# 2) Locate the existing Blind-A prediction (predicted_track_ids to KEEP).
#    Accepts a raw prediction.json OR a CodaBench submission zip.
import os, json, zipfile, glob

TOP_N = 3   # speculative: Gemini may synthesize 3 tracks (v5-kto diluted at 3, but it was
            # trained on 1; Gemini isn't). If LLM score drops vs the flash/top1 0.41 result,
            # revert to TOP_N = 1 (known-good).
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

# Your candidates file on Drive (json or zip carrying predicted_track_ids).
SRC = '/content/drive/MyDrive/recsys2026/194-union-sasrec-lgbm-cleanfull-v5kto-blindA.json'

PRED_IN  = '/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/blindA_candidates.json'
PRED_OUT = '/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/blindA_gemini.json'
os.makedirs(os.path.dirname(PRED_IN), exist_ok=True)

if not os.path.exists(SRC):
    print('SRC not found. Candidates seen under MyDrive:',
          glob.glob('/content/drive/MyDrive/**/*blindA*.*', recursive=True))
    raise FileNotFoundError(f'Set SRC to your candidates file. Looked for: {SRC}')

if SRC.endswith('.zip'):
    with zipfile.ZipFile(SRC) as zf:
        name = 'prediction.json' if 'prediction.json' in zf.namelist() else zf.namelist()[0]
        rows = json.loads(zf.read(name))
else:
    rows = json.load(open(SRC))

assert isinstance(rows, list) and all('predicted_track_ids' in r for r in rows), 'need predicted_track_ids'
json.dump(rows, open(PRED_IN, 'w'), ensure_ascii=False)
print(f'using {SRC}\n -> {len(rows)} rows (expect 80), wrote {PRED_IN}')

In [ ]:
# 3) Regenerate predicted_response over Blind-A via Gemini (track_ids untouched).
#    On any per-row API failure the script keeps that row's original response.
STRUCTURED_PERSONALITY = True   # CoT: infer mood/intent/taste axes + per-axis track fit, then reply
flag = '--structured-personality' if STRUCTURED_PERSONALITY else ''
!cd /content/recsys2026 && python -u scripts/gemini_responder.py \
    --pred {PRED_IN} --out {PRED_OUT} \
    --dataset {BLIND_DATASET} --top-n {TOP_N} --sleep 0.2 {flag}
import json
out = json.load(open(PRED_OUT))
print(f'\nrows out: {len(out)}')
print('sample response:\n', out[0]['predicted_response'][:500])

In [ ]:
# 4) Validate schema (blindA) + package the CodaBench zip (root = prediction.json).
from datetime import date
import os, sys, zipfile, shutil
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

predictions = load_prediction(PRED_OUT)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('Schema validation FAILED:')
    for e in errors[:20]: print('  -', e)
    raise SystemExit('Refusing to package — fix and rerun.')
print(f'schema OK ({len(predictions)} rows for blindA — expected 80)')

ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_194_gemini.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
out_zip = package_zip(PRED_OUT, ZIP_PATH)
with zipfile.ZipFile(out_zip) as zf:
    members = zf.namelist()
assert members == ['prediction.json'], f'wrong zip layout: {members}'
print('packaged ->', out_zip, '| contains', members)

drive_zip = f'/content/drive/MyDrive/blindset_runs/{os.path.basename(ZIP_PATH)}'
os.makedirs(os.path.dirname(drive_zip), exist_ok=True)
shutil.copy(out_zip, drive_zip)
print('Drive copy ->', drive_zip)
print('\nUpload this zip to CodaBench.')